In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

In [ ]:
def filter_output(v, num_vars=100):
    #Genes that are directly adjacent to centromeres
    blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
                "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
                "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]
    
    #Genes that are near or in IGV etc. areas
    blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP"]
    
    #Genes that seem a little iffy since they are embedded in regions with lots of the same gene
    iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
    orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
    orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
    v = v[v["NearestGene"].isin(orth["Gene name"])]
    v = v[~v["NearestGene"].isin(blacklist + blacklist2)]
    
    olfs = []
    hlas = []
    ugt = []
    for i in v["NearestGene"]:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)
    
    #Remove olfactory receptors and HLA genes
    v = v[~v["NearestGene"].isin(olfs + hlas + ugt)]
    
    #Filter to only genes with sufficient nearby substituions
    v = v[((v["Species1 Sum Total_Vars"] > num_vars) | (v["Species2 Sum Total_Vars"] > num_vars)) & ((v["Species1 Sum PhyloP NonNeg"] > 0) | (v["Species1 Sum PhyloP NonNeg"] > 0))]
    v["p-value"] = np.maximum(v["Better_Sims_Sum"]/v["Total_Sims"]*2, 1/v["Total_Sims"])
    v["FDR Sum"] = fdrcorrection(v["p-value"])[1]
    return v

In [ ]:
def filter_ont(v, num_vars = 100):
    v = v[((v["Species1 Sum Total_Vars"] > num_vars) | (v["Species2 Sum Total_Vars"] > num_vars)) & ((v["Species1 Sum PhyloP NonNeg"] > 0) | (v["Species1 Sum PhyloP NonNeg"] > 0))]
    v["p-value"] = np.maximum(v["Better_Sims_Sum"]/v["Total_Sims"]*2, 1/v["Total_Sims"])
    v["FDR Sum"] = fdrcorrection(v["p-value"])[1]
    return v

In [ ]:

def read_utr(folder):
    v = 0
    i = 0
    for file in os.listdir(folder):
        if file.endswith("0.txt"):
            if i:
                y = pd.read_csv(folder + "/" + file, sep = "\t")
                if not folder.endswith("HPO") and not folder.endswith("GOBP"):
                    y = y.set_index("NearestGene")
                else:
                    y = y.set_index("Term")
                z = v[["Total_Sims", "Better_Sims_L2FC", "Better_Sims_Sum"]] + y[["Total_Sims", "Better_Sims_L2FC", "Better_Sims_Sum"]]
                
                z = z.dropna()
                v["Total_Sims"] = z["Total_Sims"].copy()
                v["Better_Sims_L2FC"] = z["Better_Sims_L2FC"].copy()
                v["Better_Sims_Sum"] = z["Better_Sims_Sum"].copy()
            else:
                v = pd.read_csv(folder + "/" + file, sep = "\t")
                if not folder.endswith("HPO") and not folder.endswith("GOBP"):
                    v.index = v["NearestGene"]
                else:
                    v.index = v["Term"]
                i = 1
    return v

In [ ]:
def mvp(v, xlab, title, stat = "PhyloP Difference", fdr_col = "FDR"):
    plt.figure(figsize=(6, 4), dpi=600)
    sig = []
    #Also filtered out sites that were variable in apes or variable across human populations
    for index, row in v.iterrows():
        if row[fdr_col] < 0.05 and row[stat] < 0:
            sig.append("Chimp accelerated")
        elif row[fdr_col] < 0.05 and row[stat] > 0:
            sig.append("Human accelerated")
        else:
            sig.append("Not significant")
    v["Category"] = sig
    v["-Log$_{10}$(FDR)"] = -np.log10(v[fdr_col])
    v = v.sort_values("Category")
    sns.scatterplot(data = v, x = stat, y = "-Log$_{10}$(FDR)", hue = "Category", palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey"})

    xlim1 = min(v[stat]) - abs(floor(0.1*min(v[stat])))
    xlim2 = max(v[stat]) + abs(floor(0.1*max(v[stat])))
    plt.xlim(xlim1, xlim2)
    plt.hlines(-np.log10(0.05), xlim1, xlim2, linestyles="dashed", colors="k")
    plt.legend(bbox_to_anchor = (1, 1))
    plt.xlabel(xlab)
    plt.title(title)
    plt.legend([], [], frameon = False)
    plt.show()

In [ ]:
#The bias toward more human significant categories is partially the result of a global bias toward slightly higher PhyloP scores in the human lineage
v = filter_output(read_utr("UTR_Results/PhyloP_3UTR_FiltWGS_PhyloP-100_SpecSup250_NCM_NCV_Per0_PerGene"), 5).sort_values("p-value")
#v["PhyloP Difference"] = v["PhyloP Difference"] - np.mean(v["PhyloP Difference"])
#mvp(v, xlab = "Difference in PhyloP", title = "Accelerated 3' UTR evolution", fdr_col = "FDR Sum")
v

In [ ]:
v = v.drop(["PhyloP L2FC", "Total_Sims", "Species1 Sum PhyloP447", "Species2 Sum PhyloP447", "Better_Sims_L2FC", "Better_Sims_Sum"], axis = 1)
v.columns = ["Gene symbol", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]
v["UTR"] = np.repeat("3' UTR", v.shape[0])
v = v[["Gene symbol", "UTR", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]]
v3 = v.copy()

In [ ]:
v = filter_output(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene"), 5).sort_values("p-value")
v = v.drop(["PhyloP L2FC", "Total_Sims", "Species1 Sum PhyloP447", "Species2 Sum PhyloP447", "Better_Sims_L2FC", "Better_Sims_Sum"], axis = 1)
v.columns = ["Gene symbol", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]
v["UTR"] = np.repeat("5' UTR", v.shape[0])
v = v[["Gene symbol", "UTR", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]]
v5 = v.copy()
v5

In [ ]:
v = pd.concat([v3, v5]).sort_values("p-value")
k = []
for index, row in v.iterrows():
    if row["Difference in PhyloP-weighted sum"] < 0 and row["FDR"] < 0.05:
        k.append("Chimpanzee-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] < 0 and row["p-value"] < 0.05:
        k.append("Nominally chimpanzee-accelerated, p < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["FDR"] < 0.05:
        k.append("Human-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["p-value"] < 0.05:
        k.append("Nominally human-accelerated, p < 0.05")
    else:
        k.append("Not accelerated")
v["Acceleration"] = k
v.to_csv("Supplemental_Tables/Supplemental_Table3.csv", index = False)

In [ ]:
#The bias toward more human significant categories is partially the result of a global bias toward slightly higher PhyloP scores in the human lineage
v = filter_output(read_utr("UTR_Results/PhyloP_3UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene"), 5).sort_values("p-value")
v2 = filter_output(read_utr("UTR_Results/PhyloP_3UTR_NoFiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene"), 5).sort_values("p-value")

#v["PhyloP Difference"] = v["PhyloP Difference"] - np.mean(v["PhyloP Difference"])
mvp(v, xlab = "Difference in PhyloP", title = "Accelerated 3' UTR evolution", fdr_col = "FDR Sum")

In [ ]:
from scipy.stats import binomtest
ha = v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] > 0)].shape[0]
ca = v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] < 0)].shape[0]

print(ha, ca, binomtest(ha, ha + ca))

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

sns.barplot(y=["Human", "Chimp"], x=[272, 205], hue=["Human", "Chimp"], palette=palette, alpha=0.8)

plt.xlabel("Number of nominally accelerated 3' UTRs")
plt.xlim(0, 330)
#plt.xticks(size=12)
#plt.yticks(size=12)
plt.title("More human acceleration in 3' UTRs")
plt.legend([], [], frameon=False)

# --- significance bracket (horizontal) ---
y1, y2 = 0, 1        # bar positions (Human, Chimp)
x, h = 290, 5        # x-position of bracket + horizontal offset

plt.plot([x, x + h, x + h, x], [y1, y1, y2, y2], lw=1.2, c='black')

plt.text(x + 2*h, (y1 + y2) * 0.5, "**", ha='left', va='center', fontsize=14)

plt.show()

In [ ]:
from scipy.stats import binomtest
ha = len(np.intersect1d(v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] > 0)]["NearestGene"], v2[(v2["p-value"] < 0.05) & (v2["PhyloP Difference"] > 0)]["NearestGene"]))
ca = len(np.intersect1d(v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] < 0)]["NearestGene"], v2[(v2["p-value"] < 0.05) & (v2["PhyloP Difference"] < 0)]["NearestGene"]))

print(ha, ca, binomtest(ha, ha + ca))

In [ ]:
v = filter_output(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene"), 5).sort_values("p-value")
v2 = filter_output(read_utr("UTR_Results/PhyloP_5UTR_NoFiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene"), 5).sort_values("p-value")

mvp(v, xlab = "Difference in PhyloP", title = "Accelerated 5' UTR evolution", fdr_col = "FDR Sum")

In [ ]:
from scipy.stats import binomtest
ha = v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] > 0)].shape[0]
ca = v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] < 0)].shape[0]

print(ha, ca, binomtest(ha, ha + ca))

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

sns.barplot(y=["Human", "Chimp"], x=[56, 32], hue=["Human", "Chimp"], palette=palette, alpha=0.8)

plt.xlabel("Number of nominally accelerated 5' UTRs")
plt.xlim(0, 75)
#plt.xticks(size=12)
#plt.yticks(size=12)
plt.title("More human acceleration in 5' UTRs")
plt.legend([], [], frameon=False)

# --- significance bracket (horizontal) ---
y1, y2 = 0, 1        # bar positions (Human, Chimp)
x, h = 63, 1        # x-position of bracket + horizontal offset

plt.plot([x, x + h, x + h, x], [y1, y1, y2, y2], lw=1.2, c='black')

plt.text(x + 2*h, (y1 + y2) * 0.5, "*", ha='left', va='center', fontsize=14)

plt.show()

In [ ]:
plt.figure(figsize=(4, 4), dpi=600)
sns.barplot(x = ["Human", "Chimp"], y = [56, 32], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of nominally accelerated 5' UTRs", size = 12)
plt.ylim(0, 75)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("More human acceleration in 3' UTRs", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 63, 1             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "*",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
from scipy.stats import binomtest
ha = len(np.intersect1d(v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] > 0)]["NearestGene"], v2[(v2["p-value"] < 0.05) & (v2["PhyloP Difference"] > 0)]["NearestGene"]))
ca = len(np.intersect1d(v[(v["p-value"] < 0.05) & (v["PhyloP Difference"] < 0)]["NearestGene"], v2[(v2["p-value"] < 0.05) & (v2["PhyloP Difference"] < 0)]["NearestGene"]))

print(ha, ca, binomtest(ha, ha + ca))

In [ ]:
v = filter_output(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_NCM_NCV_Per0_PerGene"), 5).sort_values("p-value")
mvp(v, xlab = "Difference in PhyloP", title = "Accelerated 5' UTR evolution", fdr_col = "FDR Sum")

In [ ]:
#This bias seems to be compounded when you do ontology terms
vv = filter_ont(read_utr("UTR_Results/PhyloP_3UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_HPO"), 10).sort_values("p-value")
mvp(vv, xlab = "Difference in PhyloP", title = "3' UTR human phenotype ontology", fdr_col = "FDR Sum")
vv

In [ ]:
v = filter_ont(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_NCM_NCV_Per0_PerGene_HPO"), 10).sort_values("p-value")
mvp(v, xlab = "Difference in PhyloP", title = "5' UTR human phenotype ontology", fdr_col = "FDR Sum")

In [ ]:
v = filter_ont(read_utr("UTR_Results/PhyloP_3UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_GOBP"), 10).sort_values("p-value")
mvp(v, xlab = "Difference in PhyloP", title = "3' UTR biological process", fdr_col = "FDR Sum")

In [ ]:
v = filter_ont(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_GOBP"), 10).sort_values("p-value")
mvp(v, xlab = "Difference in PhyloP", title = "5' UTR biological process", fdr_col = "FDR Sum")

In [ ]:
v = filter_ont(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_GOBP"), 10).sort_values("p-value")
v = v.drop(["PhyloP L2FC", "Total_Sims", "Species1 Sum PhyloP447", "Species2 Sum PhyloP447", "Better_Sims_L2FC", "Better_Sims_Sum"], axis = 1)
v.columns = ["Term", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]
v["UTR"] = np.repeat("5' UTR", v.shape[0])
v = v[["Term", "UTR", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]]
k = []
for index, row in v.iterrows():
    if row["Difference in PhyloP-weighted sum"] < 0 and row["FDR"] < 0.05:
        k.append("Chimpanzee-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] < 0 and row["p-value"] < 0.05:
        k.append("Nominally chimpanzee-accelerated, p < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["FDR"] < 0.05:
        k.append("Human-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["p-value"] < 0.05:
        k.append("Nominally human-accelerated, p < 0.05")
    else:
        k.append("Not accelerated")
v["Acceleration"] = k
v.to_csv("Supplemental_Tables/Supplemental_Table4.csv", index = False)

In [ ]:
binomtest(58, 39 + 58)

In [ ]:
gobp = pd.read_csv("../DPSC_CNCC/GOBP_AccelEvol_Input.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("../DPSC_CNCC/HPO_AccelEvol_Input.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
    

In [ ]:
v = filter_output(read_utr("UTR_Results/PhyloP_5UTR_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene"), 5).sort_values("p-value")


In [ ]:
z5 = z[z["Category"] == "5UTR"]


In [ ]:
#For top two categories
#"Regulation Of Supramolecular Fiber Organization (GO:1902903)"
#"Synaptic Transmission, Glutamatergic (GO:0035249)"
#No individual gene is significant
#Only Regulation Of Supramolecular Fiber Organization (GO:1902903) beats the FDR cutoff when multiplying p-value by 2 which is I think correct
#Mann-Whitney U has much higher median non-negative PhyloP for human than chimp (one-sided p-value = 8.07e-5 for Supramolecular, 0.034 for synaptic)
#Of the Supramolecular genes, only MAP1B (aka Tau) is DE in Rachel's data
#In L2/3 IT data, CAMSAP1, CAMSAP2, MAPRE3 have lower expr from human allele, AKAP13 has higher, all of these are clearly human derived

#Extending out to all Supramolecular genes with at least one UTR variant, CHRNA7, MAP1A, ARL2 are additionally down and GBA2 is up, but unclear who is derived
z4 = z3[z3["NearestGene"].isin(d_BP["Regulation Of Supramolecular Fiber Organization (GO:1902903)"])].sort_values("PhyloP447")
mwu(z4[z4["Derived"] == "H"]["PhyloP Non"], z4[z4["Derived"] == "C"]["PhyloP Non"], alternative = "greater")

In [ ]:
#For top two categories
#"Regulation Of Supramolecular Fiber Organization (GO:1902903)"
#"Synaptic Transmission, Glutamatergic (GO:0035249)"
#No individual gene is significant
#Only Regulation Of Supramolecular Fiber Organization (GO:1902903) beats the FDR cutoff when multiplying p-value by 2 which is I think correct
#Mann-Whitney U has much higher median non-negative PhyloP for human than chimp (one-sided p-value = 8.07e-5 for Supramolecular, 0.034 for synaptic)
#Of the Supramolecular genes, only MAP1B (aka Tau) is DE in Rachel's data
#In L2/3 IT data, CAMSAP1, CAMSAP2, MAPRE3 have lower expr from human allele, AKAP13 has higher, all of these are clearly human derived

#Extending out to all Supramolecular genes with at least one UTR variant, CHRNA7, MAP1A, ARL2 are additionally down and GBA2 is up, but unclear who is derived
z4 = z3_norm[z3_norm["NearestGene"].isin(d_BP["Regulation Of Supramolecular Fiber Organization (GO:1902903)"])].sort_values("PhyloP447")
mwu(z4[z4["Derived"] == "H"]["Statistic"], z4[z4["Derived"] == "C"]["Statistic"], alternative = "greater")

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

z4["Derived"] = z4["Derived"].replace("C", "Chimp").replace("H", "Human")
sns.kdeplot(data = z4, hue = "Derived", x = "Statistic", common_norm = False, fill = True, palette = palette)
#handles, labels = ax.get_legend_handles_labels()
#ax.legend(handles=handles[1:], labels=labels[1:])
plt.xlabel("Non-negative PhyloP")
plt.title("Regulation of supramolecular fiber organization")

In [ ]:
#Main for 5' UTR is Ventricular Septum Morphogenesis (GO:0060412)
z4 = z5[z5["NearestGene"].isin(d_BP["Ventricular Septum Morphogenesis (GO:0060412)"])].sort_values("PhyloP447")
z4["PhyloP Non"] = np.maximum(z4["PhyloP447"], 0)
mwu(z4[z4["Derived"] == "H"]["PhyloP Non"], z4[z4["Derived"] == "C"]["PhyloP Non"], alternative = "greater")

In [ ]:
#Main for 5' UTR is Ventricular Septum Morphogenesis (GO:0060412)
z4 = z5[z5["NearestGene"].isin(q["Gene1"])].sort_values("PhyloP447")
z4["PhyloP Non"] = np.maximum(z4["PhyloP447"], 0)
mwu(z4[z4["Derived"] == "H"]["PhyloP Non"], z4[z4["Derived"] == "C"]["PhyloP Non"], alternative = "greater")

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

z4["Derived"] = z4["Derived"].replace("C", "Chimp").replace("H", "Human")
sns.kdeplot(data = z4, hue = "Derived", x = "PhyloP Non", common_norm = False, fill = True, palette = palette)
plt.xlabel("Non-negative PhyloP")
plt.title("Ventricular septum morphogenesis")


In [ ]:
#Almost all the Ventricular Septum Morphogenesis genes are downregulated in CMs which is pretty cool and consistent with their accelerated 5' UTR evolution
x = pd.read_csv("../Myriad_RNA/CM_Filtered.txt", sep = "\t")
x2 = x[x["gene"].isin(d_BP["Ventricular Septum Morphogenesis (GO:0060412)"])]
x2 = x2[x2["padj Chpreffed"] < 0.05]
up = x2[x2["L2FC Chpreffed"] > 0].shape[0]
down = x2[x2["L2FC Chpreffed"] < 0].shape[0]
x3 = x[x["padj Chpreffed"] < 0.05]
from scipy.stats import binomtest
binomtest(down, up + down, p = x3[x3["L2FC Chpreffed"] < 0].shape[0]/(x3[x3["L2FC Chpreffed"] < 0].shape[0] + x3[x3["L2FC Chpreffed"] > 0].shape[0]))

In [ ]:

sns.set(font_scale = 1.4)
sns.set_style("white")
plt.figure(figsize=(3.5,5), dpi=600)
t_ax = sns.barplot({"Upregulated\nin human":1, "Downregulated\nin human":13}, errorbar=None, linewidth=2.5, edgecolor=".5", facecolor='#F2C91140', gap = 0.1, palette = {"Upregulated\nin human":"#CD99D8", "Downregulated\nin human":"#39BEB1"})
for patch in t_ax.patches:
    print(patch.get_x() + patch.get_width()/2)
c = 0
for patch in t_ax.patches:
    if c < 1:
        patch.set_edgecolor("#CD99D8")
        patch.set_facecolor("#CD99D8" + "80")
    elif c == 1:
        patch.set_edgecolor("#39BEB1")
        patch.set_facecolor("#39BEB1" + "80")
    c += 1
plt.ylabel("Number of genes")
plt.title("Septum morphogensis\ngenes in cardiomyocytes")
plt.ylim(0, 16)
x1, x2 = 0, 1                 # bar positions
y, h = 14, 0.4             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "**",
         ha='center', va='bottom', fontsize=14)

In [ ]:
#Notably, 7 out of 9 of the genes that might be associated with trabeculation are predicted to result in less trabeculation with decrease expression!

to_plot = x2.sort_values("L2FC Chpreffed")
out = []
for index, row in to_plot.iterrows():
    #out.append(["$_{" + index + "}$", row["log2FoldChange HumChp"], "Hybrid cortical organoid"])
    #out.append(["$_{" + index + "}$", row["log2FoldChange HumChp Adult"], "Adult MTG L2/3 IT"])
    if row["L2FC Chpreffed"] < 0:
        x = "Down human"
    else:
        x = "Up human"
    out.append(["$_{" + row["gene"] + "}$", row["L2FC Chpreffed"], x])
to_plot = pd.DataFrame(out)
to_plot.columns = ["Gene", "Log$_{2}$ fold-change", "Direction"]
sns.set(font_scale = 1.5)
sns.set_style("white")
plt.figure(figsize=(7,4), dpi=600)
sns.set_style("white")
new_palette = palette
t_ax = sns.barplot(data = to_plot, x = "Gene", y = "Log$_{2}$ fold-change", hue = "Direction")


c = 0
for patch in t_ax.patches:
    if c < 13:
        patch.set_edgecolor("#39BEB1")
        patch.set_facecolor("#39BEB1" + "80")
    else:
        patch.set_edgecolor("#CD99D8")
        patch.set_facecolor("#CD99D8" + "80")
    c += 1
plt.title("Septum morphogenesis genes in cardiomyocytes", size = 15)
plt.xticks(rotation=90)
plt.ylabel("Log$_{2}$ fold-change human/chimp", size = 14)
plt.legend([], [], frameon = False)